**Exercício 6**

Usando pacote TensorFlow, implemente uma rede convolucional para classificar a base de imagens CIFAR-10.

Utilize pelo menos 3 camadas convolucionais com e sem dropout.

Compare os resultados obtidos (acurácia) usando os algoritmos: ADAGRAD, ADAGRAD + dropout, RMS-PROP, RMS-PROP + dropout, ADAM, ADAM +dropout.

In [48]:
import tensorflow as tf

from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adagrad, RMSprop, Adam
import matplotlib.pyplot as plt

Carregando a base de dados CIFAR-10

In [49]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

Construindo modelo CNN

In [50]:
def cnn_model(dropout=False):
  model = models.Sequential()
  model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
  model.add(layers.MaxPooling2D((2, 2)))
  model.add(layers.Conv2D(64, (3, 3), activation='relu'))
  model.add(layers.MaxPooling2D((2, 2)))
  model.add(layers.Conv2D(64, (3, 3), activation='relu'))

  model.add(layers.Flatten())
  model.add(layers.Dense(64, activation='relu'))

  # Adicionar Dropout, se aplicável
  if dropout:
    model.add(layers.Dropout(0.5))

  model.add(layers.Dense(10, activation='softmax'))

  return model

Função de treino do modelo

In [51]:
def train_model(optimizer, dropout=False):
  model = cnn_model(dropout)
  model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

  history = model.fit(train_images, train_labels, epochs=10,
                      validation_data=(test_images, test_labels))

  test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)

  return test_acc, history

Resultados

In [52]:
def plot_accuracy(history, name):
  plt.plot(history.history['accuracy'], label='accuracy')
  plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.ylim([0., 1])
  plt.legend(loc='lower right')
  plt.title(name)
  plt.savefig(name+'.png')
  plt.close()

optimizers = [
    ("Adagrad", Adagrad()),
    ("Adagrad + Dropout", Adagrad(), True),
    ("RMSprop", RMSprop()),
    ("RMSprop + Dropout", RMSprop(), True),
    ("Adam", Adam()),
    ("Adam + Dropout", Adam(), True)
]

results = {}
for name, optimizer, *dropout in optimizers:
    dropout_flag = dropout[0] if dropout else False
    acc, history = train_model(optimizer, dropout_flag)
    plot_accuracy(history, name)
    results[name] = acc
    print(f"{name}: Acurácia = {acc}")

# Exibir os resultados
for key, value in results.items():
    print(f"{key}: Acurácia = {value}")

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 67s 42ms/step - accuracy: 0.1132 - loss: 2.2951 - val_accuracy: 0.1776 - val_loss: 2.2565
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 67s 43ms/step - accuracy: 0.2084 - loss: 2.2132 - val_accuracy: 0.2659 - val_loss: 2.0458
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 69s 44ms/step - accuracy: 0.2877 - loss: 1.9946 - val_accuracy: 0.3240 - val_loss: 1.9041
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 81s 43ms/step - accuracy: 0.3261 - loss: 1.8885 - val_accuracy: 0.3521 - val_loss: 1.8232
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 81s 43ms/step - accuracy: 0.3558 - loss: 1.8181 - val_accuracy: 0.3726 - val_loss: 1.7677
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 84s 44ms/step - accuracy: 0.3749 - loss: 1.7601 - val_accuracy: 0.3929 - val_loss: 1.7155
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 43ms/step - accuracy: 0.3905 - loss: 1.7117 - val_accuracy: 0.3993 - val_loss: 1.6836
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.4049 -